# FINS notebook 13 — a model of the tracer alone with a learned hidden field (exp70 step 2; Colab, GPU)

Registered in `README_EXPERIMENTS.md` (exp70 step 2, 11.10) before any of these runs. Step 2 of exp70: no simulator and
no hidden variable in the data. A model of c alone with a learned hidden field z (LatentModel: an encoder of four
snapshots of c gives z, a stepper advances (z, c)), trained on c only (exp69's window emulators' training set), seeds
42-44 at 128^2 (1024 trajectories, 10000 steps of 32, 40 modes, loss on c over four steps; about 1.5-2 h each on a T4).
FINS through its tangent with z in the hidden variable's place: arm data, arm self (its own rollouts), arm lift (the
best perturbation of z). A few minutes per arm.
* cell 2 trains the three models (resumable); cell 3 runs the arms; cell 4 shows progress; cell 5 zips the json and
  logs to `fins_nb13.zip` (small: upload this one to the chat).

Independent of notebook 12 (run either first, or both in one session). Every trainer writes `<checkpoint>.partial` to
Drive every 5000 steps and resumes from it: after a disconnect, rerun cell 1 and the cell that was running. Drive needs
about 1.5 GB free (a latent checkpoint is about 0.2 GB, 0.6 GB while training); files that Colab deletes or overwrites go to the Trash and still count, so empty the Trash first.

Before running, upload the newest **`fins_repo_full.zip`** from the chat to My Drive (root or one folder down).

In [ ]:
# 1. setup: Drive, the newest repository zip (with exp70), neuraloperator, pysindy and pyqg, checks, GPU, Drive space
import os, sys, subprocess, zipfile, glob, time, shutil
from google.colab import drive
drive.mount('/content/drive')
os.environ['FINS_RESULTS_DIR'] = '/content/drive/MyDrive/fins_runs/results'
os.environ['TQDM_DISABLE'] = '1'
R = os.environ['FINS_RESULTS_DIR']
os.makedirs(os.path.join(R, 'tracer'), exist_ok=True)


def newest(pattern, test):
    cands = sorted(glob.glob(f'/content/drive/MyDrive/{pattern}') + glob.glob(f'/content/drive/MyDrive/*/{pattern}'),
                   key=os.path.getmtime, reverse=True)
    for z in cands:
        ok = test(z)
        print(f'{z}  {time.ctime(os.path.getmtime(z))}  {os.path.getsize(z)} bytes  usable: {ok}')
        if ok:
            return z


def has_exp70(z):
    with zipfile.ZipFile(z) as zz:
        names = zz.namelist()
        return ('fins_repo/experiments/exp70_learned_hidden.py' in names
                and b'def plane_readouts' in zz.read('fins_repo/experiments/exp70_learned_hidden.py'))


ZIP_PATH = newest('fins_repo_full*.zip', has_exp70)
if ZIP_PATH is None:
    raise SystemExit('no fins_repo_full*.zip on Drive has exp70 step 2: upload the newest zip from the chat')
print('using', ZIP_PATH)
shutil.rmtree('/content/fins', ignore_errors=True)
with zipfile.ZipFile(ZIP_PATH) as z:
    z.extractall('/content/fins')
EXPDIR = '/content/fins/fins_repo/experiments'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'neuraloperator', 'pysindy',
                'git+https://github.com/pyqg/pyqg.git'], check=False)
chk = subprocess.run([sys.executable, 'exp70_learned_hidden.py', 'check', '--n', '64'], cwd=EXPDIR, capture_output=True,
                     text=True)
print(chk.stdout[-2500:], chk.stderr[-1500:])
if chk.returncode:
    raise SystemExit('exp70 check failed')
import torch
print('torch', torch.__version__, '| cuda:', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
if not torch.cuda.is_available():
    print('NO GPU: Runtime -> Change runtime type -> T4 GPU (or better), then rerun this cell')
total, used, free = shutil.disk_usage('/content/drive/MyDrive')
print(f'Drive: {free / 2 ** 30:.2f} GB free of {total / 2 ** 30:.1f} GB (needs about 1.5 GB)')
CACHE = '/content/tracer_cache'
RUNS = os.path.join(R, 'tracer70_runs')
os.makedirs(os.path.join(RUNS, 'logs'), exist_ok=True)
SEEDS = (42, 43, 44)
COMMON = ['--n', '128', '--cache', CACHE]
TRAIN = ['--modes', '40', '--n-traj', '1024', '--steps', '10000', '--batch', '32', '--log-every', '500',
         '--ckpt-every', '5000']
KEYS = ('held-out error', 'own rollouts', 'pencil', 'lowest', 'observed:', 'heat-false', 'H (10', 'lift s')


def go(stem, cmd, keys=KEYS):
    if os.path.exists(stem + '.json'):
        print('done', os.path.basename(stem))
        return
    log = os.path.join(RUNS, 'logs', os.path.basename(stem) + '.log')
    t0 = time.time()
    with open(log, 'a') as fh:
        fh.write('$ ' + ' '.join(cmd) + '\n')
        fh.flush()
        rc = subprocess.call([sys.executable, '-u', *cmd], cwd=EXPDIR, stdout=fh, stderr=subprocess.STDOUT)
    lines = open(log).read().splitlines()
    keep = [l.strip() for l in lines if any(k in l for k in keys)] if rc == 0 else lines[-8:]
    print(os.path.basename(stem), 'exit', rc, f'{time.time() - t0:.0f} s', *[k[:170] for k in keep[-12:]], sep='\n   ')


def train(emu, s, extra=()):
    log = os.path.join(RUNS, 'logs', f'train_{emu}_s{s}.log')
    t0 = time.time()
    print(f'{emu} s{s}: training (log {log})', flush=True)
    with open(log, 'a') as fh:
        cmd = ['exp70_learned_hidden.py', 'train', '--emu', emu, '--seed', str(s), *COMMON, *TRAIN, *extra]
        fh.write('$ ' + ' '.join(cmd) + '\n')
        fh.flush()
        p = subprocess.Popen([sys.executable, '-u', *cmd], cwd=EXPDIR, stdout=fh, stderr=subprocess.STDOUT)
        seen = len(open(log).read().splitlines())
        while p.poll() is None:
            time.sleep(60)
            lines = open(log).read().splitlines()
            for l in lines[seen:]:
                print('   ', l.strip()[:170], flush=True)
            seen = len(lines)
    lines = open(log).read().splitlines()
    print(f'{emu} s{s}: exit {p.returncode}, {time.time() - t0:.0f} s', *[l[:170] for l in lines[seen:]], sep='\n   ')


def ckpt(emu, s):
    name = 'state_n128_k0.05_m40' if emu == 'state' else 'latent_n128_k0.05_m40_w4_r4'
    return os.path.join(R, 'tracer', f'{name}_s{s}.pt')


ARMS = (('data', ['--arm', 'data', '--fp64']), ('self', ['--arm', 'self', '--fp64']),
        ('lift', ['--arm', 'lift', '--fp64', '--obs-pairs', '4', '--cg-iters', '100']))
for emu in ('state', 'latent'):
    print(emu, {s: os.path.exists(ckpt(emu, s)) for s in SEEDS})

In [ ]:
# 2. train the latent models of c alone (resumable)
for s in SEEDS:
    if os.path.exists(ckpt('latent', s)):
        print('trained', os.path.basename(ckpt('latent', s)))
        continue
    train('latent', s, ['--w', '4', '--roll', '4'])
left = [f's{s}' for s in SEEDS if not os.path.exists(ckpt('latent', s))]
print('all three latent models trained' if not left else 'NOT YET: ' + ', '.join(left))

In [ ]:
# 3. FINS through each latent model: arms data, self, lift (float64 on the GPU)
for s in SEEDS:
    if not os.path.exists(ckpt('latent', s)):
        print('not trained yet: latent s', s)
        continue
    for arm, extra in ARMS:
        stem = os.path.join(RUNS, f'ex70_latent_{arm}_s{s}')
        go(stem, ['exp70_learned_hidden.py', 'fins', '--emu', 'latent', '--w', '4', '--roll', '4', *extra, '--seed',
                  str(s), *COMMON, '--save', stem])

In [ ]:
# 4. progress
for s in SEEDS:
    have = [a for a, _ in ARMS if os.path.exists(os.path.join(RUNS, f'ex70_latent_{a}_s{s}.json'))]
    print(f'latent s{s}: trained {os.path.exists(ckpt("latent", s))}; arms done:', ', '.join(have) or 'none')
for f in sorted(glob.glob(os.path.join(R, 'tracer', 'latent*.partial'))):
    print('in progress:', os.path.basename(f), time.ctime(os.path.getmtime(f)))

In [ ]:
# 5. copy the latent json and logs file by file to local disk (remount if Drive drops); fins_nb13.zip in /content and on Drive
local = '/content/fins_nb13'
shutil.rmtree(local, ignore_errors=True)
files = glob.glob(os.path.join(RUNS, 'ex70_latent_*.json')) + glob.glob(os.path.join(RUNS, 'logs', '*latent*.log'))
for f in sorted(set(files)):
    dst = os.path.join(local, os.path.relpath(f, R))
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    for attempt in range(3):
        try:
            shutil.copy2(f, dst)
            break
        except OSError as e:
            print('retry', f, e)
            drive.mount('/content/drive', force_remount=True)
            time.sleep(5)
out = '/content/fins_nb13.zip'
with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in sorted(glob.glob(os.path.join(local, '**', '*'), recursive=True)):
        if os.path.isfile(f):
            z.write(f, os.path.relpath(f, local))
print(out, os.path.getsize(out), 'bytes')
try:
    shutil.copy2(out, '/content/drive/MyDrive/fins_nb13.zip')
    print('copied to MyDrive: fins_nb13.zip (upload this one to the chat)')
except OSError as e:
    print('Drive copy failed; download /content/fins_nb13.zip from the file pane:', e)